In [1]:
!nvidia-smi


Sun Oct  4 04:43:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/Maheendra-mj/3-day-inference-challenge

Cloning into '3-day-inference-challenge'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 70 (delta 16), reused 66 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 53.59 KiB | 3.83 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [3]:
%cd 3-day-inference-challenge

/kaggle/working/3-day-inference-challenge


In [4]:
!bash scripts/kaggle_bootstrap.sh

index, name, memory.total [MiB], driver_version
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for infer-lab (pyproje

In [5]:
!infer-lab bench --config configs/smoke.yaml --tag smoke


2026-10-04 04:49:24,327 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-04 04:49:24,454 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.77MB/s]
tokenizer_config.json: 7.30kB [00:00, 16.7MB/s]
vocab.json: 2.78MB [00:00, 52.4MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 7.03MB [00:00, 142MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 988M/988M [00:03<00:00, 271MB/s]
Loading weights: 100%|█| 290/290 [00:00<00:00, 531.98it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.21MB/s]
2026-10-04 04:49:39,885 INFO infer_lab.bench.runner: run 1  bs=1   pl=64    ol=32    tok/s=    32.7  ttft_p50=   33.5ms  tpot_p50= 30.46ms  util= 40%  power=35.7W  sm

In [10]:
!infer-lab bench --config configs/bench.yaml --tag baseline


2026-10-03 16:55:34,463 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-03 16:55:34,646 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 555.27it/s, Materializing param=m
2026-10-03 16:55:56,449 INFO infer_lab.bench.runner: run 3  bs=1   pl=128   ol=128   tok/s=    28.9  ttft_p50=   39.4ms  tpot_p50= 34.46ms  util= 39%  power=33.2W  sm_clk= 699MHz
2026-10-03 16:56:13,316 INFO infer_lab.bench.runner: run 4  bs=1   pl=512   ol=128   tok/s=    30.7  ttft_p50=   49.6ms  tpot_p50= 31.84ms  util= 35%  power=37.4W  sm_clk=1124MHz
2026-10-03 16:56:30,466 INFO infer_lab.bench.runner: run 5  bs=4   pl=128   ol=128   tok/s=   119.6  ttft_p50=   37.6ms  tpot_p50= 33.34ms  util= 36%  power=36.0W  sm_clk=1072MHz
2026-10-03 16:56:47,944 INFO infe

In [6]:
!infer-lab kernels


vector_add  torch         n=67108864     3.307 ms    243.5 GB/s  ( 76.1% of T4 peak)
vector_add  scalar        n=67108864     3.594 ms    224.0 GB/s  ( 70.0% of T4 peak)
vector_add  vec4          n=67108864     3.412 ms    236.0 GB/s  ( 73.8% of T4 peak)
matmul      torch_cublas  n=512          0.069 ms   3.865 TFLOPS  ( 47.7% of T4 peak)
matmul      naive         n=512          0.453 ms   0.593 TFLOPS  (  7.3% of T4 peak)
matmul      tiled16       n=512          0.290 ms   0.924 TFLOPS  ( 11.4% of T4 peak)
matmul      torch_cublas  n=1024         0.365 ms   5.891 TFLOPS  ( 72.7% of T4 peak)
matmul      naive         n=1024         4.730 ms   0.454 TFLOPS  (  5.6% of T4 peak)
matmul      tiled16       n=1024         2.787 ms   0.770 TFLOPS  (  9.5% of T4 peak)
matmul      torch_cublas  n=2048         4.650 ms   3.695 TFLOPS  ( 45.6% of T4 peak)
matmul      naive         n=2048        40.780 ms   0.421 TFLOPS  (  5.2% of T4 peak)
matmul      tiled16       n=2048        26.832 ms   0.640

In [12]:
!infer-lab report


2026-10-02 08:24:50,250 INFO numexpr.utils: NumExpr defaulting to 4 threads.
 id      tag  backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms
  1    smoke hf_torch           1          64          32                34.25        37.17        37.17        28.92      933.59
  2    smoke hf_torch           4          64          32               132.33        31.79        31.79        30.15      966.56
  3 baseline hf_torch           1         128         128                33.76        33.36        33.66        29.44     3842.15
  4 baseline hf_torch           1         512         128                33.79        46.67        47.11        29.59     3826.54
  5 baseline hf_torch           4         128         128               137.84        32.15        33.54        29.09     3736.30
  6 baseline hf_torch           4         512         128               132.42       175.42       175.77        28.96     3909.17
  7 baseline 

In [7]:
!infer-lab profile --bs 32 --pl 128 512 --ol 32 --trace


2026-10-04 04:49:59,378 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 535.75it/s, Materializing param=m
2026-10-04 04:50:02,412 INFO infer_lab.bench.profile: profiling bs=32 pl=128 ol=32
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
2026-10-04 04:50:43,109 INFO infer_lab.bench.profile: profiling bs=32 pl=512 ol=32

 bs    pl | prefill wall  GPU busy | decode wall/step GPU busy/step GPU idle kernels/step avg kernel | verdict
 32   128 |      241.3ms   231.1ms |          29.64ms       20.97ms      29%         1421     14.8us | GPU-bound  (CPU stalled on ful

In [8]:
!cat results/profile/top_ops_bs32_pl512_ol32.txt


----------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                                  Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
----------------------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                           decode_step         0.00%       0.000us         0.00%       0.000us       0.000us        1.738s        55.71%        1.738s      56.070ms            31  
                                                               prefill         0.00%       0.000us         0.00%       0.000us       0.000us       

In [9]:
!infer-lab bench --config configs/bench.yaml --tag baseline_v2


2026-10-04 04:51:48,484 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch
2026-10-04 04:51:48,622 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 601.41it/s, Materializing param=m
2026-10-04 04:52:07,634 INFO infer_lab.bench.runner: run 3  bs=1   pl=128   ol=128   tok/s=    32.7  ttft_p50=   33.0ms  tpot_p50= 30.48ms  util= 38%  power=38.5W  sm_clk= 885MHz
2026-10-04 04:52:23,253 INFO infer_lab.bench.runner: run 4  bs=1   pl=512   ol=128   tok/s=    33.0  ttft_p50=   47.7ms  tpot_p50= 30.35ms  util= 35%  power=46.1W  sm_clk=1291MHz
2026-10-04 04:52:38,697 INFO infer_lab.bench.runner: run 5  bs=4   pl=128   ol=128   tok/s=   134.8  ttft_p50=   33.5ms  tpot_p50= 29.69ms  util= 36%  power=47.7W  sm_clk=1362MHz
2026-10-04 04:52:55,097 INFO infe

In [13]:
!infer-lab profile --bs 1 32 --pl 128 512 --ol 32


2026-10-04 05:13:16,387 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 290/290 [00:00<00:00, 504.02it/s, Materializing param=m
2026-10-04 05:13:18,451 INFO infer_lab.bench.profile: profiling bs=1 pl=128 ol=32
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
2026-10-04 05:13:56,518 INFO infer_lab.bench.profile: profiling bs=1 pl=512 ol=32
2026-10-04 05:14:37,801 INFO infer_lab.bench.profile: profiling bs=32 pl=128 ol=32
2026-10-04 05:15:20,625 INFO infer_lab.bench.profile: profiling bs=32 pl=512 ol=32

 bs    pl | prefill wall  GPU busy | decode wall/step GPU busy/step GPU idle ker

In [14]:
!infer-lab report


 id         tag  backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms  torch_peak_mem_mb  gpu.gpu0.util_mean_pct  gpu.gpu0.power_mean_w
  1       smoke hf_torch           1          64          32                32.70        33.52        33.52        30.46      977.84             962.28                   39.60                  35.75
  2       smoke hf_torch           4          64          32               135.70        32.98        32.98        29.34      942.49             974.24                   40.67                  35.70
  3 baseline_v2 hf_torch           1         128         128                32.67        33.03        38.20        30.48     3958.42             966.26                   38.30                  38.55
  4 baseline_v2 hf_torch           1         512         128                32.95        47.73        48.98        30.35     3904.06            1008.94                   35.07                  46.11
  5 b

In [12]:
import pandas as pd
df = pd.read_csv("results/telemetry/1.csv")   # use the id printed in the bench log
df.plot(x="t_s", y=["util_pct", "power_w", "sm_clock_mhz"], subplots=True, figsize=(10, 6))


FileNotFoundError: [Errno 2] No such file or directory: 'results/telemetry/1.csv'